In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
import matplotlib.pyplot as plt


In [12]:
# ================================
# 1) BASIC SETTINGS
# ================================
IMG_SIZE = (224, 224)     # MobileNetV3 standard input size
BATCH_SIZE = 32           # Number of images processed per batch
EPOCHS = 10               # Training epochs (you can increase later)

In [13]:
# IMPORTANT: Change these paths to your dataset locationtrain_dir = r".../classification_dataset/train"
train_dir = "dataset/classification_dataset/train"
val_dir   = "dataset/classification_dataset/val"
test_dir  = "dataset/classification_dataset/test"


In [14]:
# ================================
# 2) LOAD DATASET (Train + Validation)
# ================================
# We create validation set automatically from Train folder using validation_split

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

Found 46002 files belonging to 2 classes.
Found 9855 files belonging to 2 classes.
Found 9866 files belonging to 2 classes.


In [35]:
# ================================
# 4) PERFORMANCE OPTIMIZATION
# ================================
# Prefetch improves training speed
class_names = train_ds.class_names 
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [16]:
# ================================
# 5) DATA AUGMENTATION
# ================================
# This helps model generalize better by creating random variations
data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [17]:
# ================================
# 6) LOAD PRE-TRAINED MOBILENETV3 BASE MODEL
# ================================
# include_top=False means we remove the ImageNet classification head
base_model = MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

In [18]:
# Freeze base model first (Stage 1 training)
base_model.trainable = False


In [19]:

# ================================
# 7) BUILD OUR CUSTOM CLASSIFIER MODEL
# ================================
inputs = layers.Input(shape=(224, 224, 3))# this create a tensor for later receiving the image

# Apply augmentation
x = data_aug(inputs)

# Preprocess input for MobileNetV3 (important)
x = preprocess_input(x)

# Pass through MobileNetV3 feature extractor
x = base_model(x, training=False)

# Convert feature maps into a single feature vector
x = layers.GlobalAveragePooling2D()(x)

# Dropout reduces overfitting
x = layers.Dropout(0.3)(x)

# Final output layer:
# 1 neuron + sigmoid => binary classification (fresh vs rotten)
outputs = layers.Dense(1, activation="sigmoid")(x)

# Create final model
model = models.Model(inputs, outputs)


In [30]:

# ================================
# 8) COMPILE MODEL
# ================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           961 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,997,313 (11.43 MB)

 Trainable params: 961 (3.75 KB)

 Non-trainable params: 2,996,352 (11.43 MB)

In [22]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",        # watch validation loss
    patience=3,                # wait 3 epochs before stopping
    restore_best_weights=True  # go back to best model
)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# ADD THIS LINE (fix for class_names error)
class_names = ['fresh', 'rotten']

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    preds = (preds >= 0.5).astype(int).flatten()

    y_pred.extend(preds)
    y_true.extend(labels.numpy())

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


Class weights: {np.int64(0): np.float64(0.7518632322175732), np.int64(1): np.float64(1.4926022063595068)}


In [31]:
# ================================
# 9) TRAIN MODEL (STAGE 1)
# ================================
# Train only the top layers (base_model is frozen)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight
)

Epoch 1/10
1438/1438 ━━━━━━━━━━━━━━━━━━━━ 483s 332ms/step - accuracy: 0.9458 - loss: 0.1575 - val_accuracy: 0.9591 - val_loss: 0.1194
Epoch 2/10
1438/1438 ━━━━━━━━━━━━━━━━━━━━ 475s 330ms/step - accuracy: 0.9443 - loss: 0.1571 - val_accuracy: 0.9598 - val_loss: 0.1187
Epoch 3/10
1438/1438 ━━━━━━━━━━━━━━━━━━━━ 457s 318ms/step - accuracy: 0.9466 - loss: 0.1528 - val_accuracy: 0.9601 - val_loss: 0.1179
Epoch 4/10
1438/1438 ━━━━━━━━━━━━━━━━━━━━ 455s 316ms/step - accuracy: 0.9460 - loss: 0.1566 - val_accuracy: 0.9595 - val_loss: 0.1178
Epoch 5/10
1438/1438 ━━━━━━━━━━━━━━━━━━━━ 453s 315ms/step - accuracy: 0.9463 - loss: 0.1543 - val_accuracy: 0.9604 - val_loss: 0.1172
Epoch 6/10
1438/1438 ━━━━━━━━━━━━━━━━━━━━ 455s 316ms/step - accuracy: 0.9468 - loss: 0.1544 - val_accuracy: 0.9597 - val_loss: 0.1175
Epoch 7/10
1438/1438 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - accuracy: 0.9465 - loss: 0.1529

KeyboardInterrupt: 

In [32]:
# ================================
# 10) EVALUATE ON TEST DATA
# ================================
test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)

309/309 ━━━━━━━━━━━━━━━━━━━━ 74s 239ms/step - accuracy: 0.9635 - loss: 0.1149
Test Accuracy: 0.963511049747467


In [33]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Collect true labels and predictions
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    preds = (preds >= 0.5).astype(int).flatten()  # sigmoid → 0/1
    
    y_pred.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Print class order
print("Class order:", test_ds.class_names)

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# Precision, Recall, F1-score
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=test_ds.class_names
))


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 

AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [26]:
# 12) PLOT ACCURACY vs EPOCHS
# ================================
plt.figure()
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epochs")
plt.legend()
plt.show()

# ================================
# 13) PLOT LOSS vs EPOCHS
# ================================
plt.figure()
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss vs Epochs")
plt.legend()
plt.show()

NameError: name 'history' is not defined

<Figure size 640x480 with 0 Axes>

In [27]:
# ================================
# 11) SAVE MODEL
# ================================
model.save("testingmodel2.keras")
print("Model saved as apple_fresh_rotten_mobilenetv3.keras")

Model saved as apple_fresh_rotten_mobilenetv3.keras


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

# -----------------------------
# 1) Load your saved model
# -----------------------------
model = tf.keras.models.load_model("testingmodel1.keras",compile=False)

# -----------------------------
# 2) Load image using OpenCV
# -----------------------------
img_path = r"C:/Users/NINJA/Desktop/Testing/test1.jpg"
img = cv2.imread(img_path)

if img is None:
    print("Image not found! Check path.")
    exit()

# -----------------------------
# 3) Preprocess image
# -----------------------------
img = cv2.resize(img, (224, 224))              # Resize
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)     # Convert BGR to RGB

img_array = np.expand_dims(img, axis=0)        # Shape: (1,224,224,3)
img_array = preprocess_input(img_array)        # MobileNetV3 preprocessing

# -----------------------------
# 4) Predict
# -----------------------------
pred = model.predict(img_array)[0][0]

# -----------------------------
# 5) Show result
# -----------------------------
if pred >= 0.5:
    label = "Rotten"
else:
    label = "Fresh"

print("Prediction:", label, "Confidence:", pred)

# -----------------------------
# 6) Display image with label
# -----------------------------
display_img = cv2.imread(img_path)
cv2.putText(display_img, f"{label} ({pred:.2f})", (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

cv2.imshow("Prediction", display_img)
cv2.waitKey(0)
cv2.destroyAllWindows()


c:\Users\NINJA\miniconda3\envs\ml\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Image not found! Check path.


error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


: 